# Phase 5 on Google Colab: generator transfer matrix for conformal hallucination filters

**Before you start:** Runtime -> Change runtime type -> **T4 GPU** (or L4 / A100 with Colab Pro). Then Runtime -> Run all.

**How it keeps your work.** Everything finished is copied to your Google Drive (`MyDrive/Phase5_Colab/`) every 5 minutes.
If Colab disconnects, open the notebook again and press Run all: it copies the finished work back and carries on
where it stopped. At most about 5 minutes of work is lost.

**Order of work**
1. **Pilot** (`MODE = "pilot"` in cell 2): 100 images for each of the six generators, about 3 to 4 h on one T4.
   Cell 8 prints the hallucination rate (CHAIR) and truncation for each generator. Send that output for checking.
2. **Wave 1** (`MODE = "full"`, `WAVE = 1`: LLaVA, Qwen2.5-VL-3B, InternVL3-2B). About 45 h of T4 time in total, so
   several sessions. Just run the notebook again each time.
3. **Wave 2** (`WAVE = 2`: sampled LLaVA, Qwen3-VL-4B, InstructBLIP). About 50 h of T4 time.
4. When cell 7 says COMPLETE, cell 9 runs the analysis and the audit sample.

**Colab and Kaggle together.** Each wave writes its own files, so Kaggle can run one wave while Colab runs the other.
Do not run the **same** wave on both. At the end, put all `*.jsonl` files into one folder and run cell 9 there.

**Before cell 9 runs on full data, the pre-registration must be frozen** (`prereg-v3` git tag) and `REF` in cell 3
set to `"prereg-v3"`.

In [ ]:
# 1. GPU check and Google Drive
import os, subprocess, sys, time, glob, json, shutil
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "NO GPU: Runtime -> Change runtime type -> T4 GPU")
import torch
NGPU = torch.cuda.device_count()
assert NGPU > 0, "No GPU. Runtime -> Change runtime type -> T4 GPU, then run again."
from google.colab import drive
drive.mount("/content/drive")
print("GPUs:", NGPU, "|", torch.cuda.get_device_name(0))

In [ ]:
# 2. Settings: change only these
MODE = "pilot"          # "pilot" (100 images per generator) or "full"
WAVE = 1                # 1 = llava, qwen25, internvl3 ; 2 = llava_t1, qwen3_4b, instructblip
SESSION_HOURS = 11.0    # stop GPU work after this long (free Colab sessions last up to ~12 h)
WAVES = {1: "llava,qwen25,internvl3", 2: "llava_t1,qwen3_4b,instructblip"}
GENS = WAVES[WAVE] if MODE == "full" else WAVES[1] + "," + WAVES[2]
NAME = "phase5" if MODE == "full" else "phase5_pilot"
WORK = "/content"
OUT = os.path.join(WORK, NAME)                                     # fast local disk
SAVE = os.path.join("/content/drive/MyDrive/Phase5_Colab", NAME)   # permanent copy on Drive
CACHE = "/content/drive/MyDrive/Phase5_Colab/cache"                # AMBER image zip, kept between sessions
for d in (OUT, SAVE, CACHE):
    os.makedirs(d, exist_ok=True)
LIMIT = ["--limit", "100"] if MODE == "pilot" else []
os.environ["HF_HOME"] = "/content/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(MODE, "| generators:", GENS, "\nwork:", OUT, "\nsaved to:", SAVE)

In [ ]:
# 3. Code and packages. After the pre-registration is frozen, set REF = "prereg-v3".
REPO = "https://github.com/antonyjasfer/Budget-Coupled-Robust-Belief-Propagation-for-Visual-Hallucination-Detection.git"
REF = "q1-tai-upgrade"
os.chdir(WORK)
if os.path.exists("bcrbp"):
    subprocess.run(["git", "-C", "bcrbp", "pull", "-q"], check=False)    # pick up fixes pushed since last session
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", REF, REPO, "bcrbp"], check=True)
os.chdir(os.path.join(WORK, "bcrbp"))
!pip -q install inflect bitsandbytes accelerate gdown
import transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
!git log --oneline -1

def save_to_drive():
    """Copy finished work (JSONL rows, logs, results) from local disk to Drive."""
    n = 0
    for f in (glob.glob(f"{OUT}/*.jsonl") + glob.glob(f"{OUT}/*.log") + glob.glob(f"{OUT}/*.json")
              + glob.glob(f"{OUT}/audit/*.csv") + glob.glob(f"{OUT}/*.zip")):
        dst = os.path.join(SAVE, os.path.relpath(f, OUT))
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(f):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(f, dst)
            n += 1
    return n

def restore_from_drive():
    """Copy earlier sessions' rows back; a half-written last line is dropped so appending stays clean."""
    n = 0
    for f in glob.glob(f"{SAVE}/*.s*.jsonl"):
        dst = os.path.join(OUT, os.path.basename(f))
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(f):
            text = open(f, encoding="utf-8").read()
            if text and not text.endswith("\n"):
                text = text[: text.rfind("\n") + 1]
            open(dst, "w", encoding="utf-8").write(text)
            n += 1
    return n

In [ ]:
# 4. Restore earlier sessions, setup (annotations, seeded split), images (~15-25 min, CPU)
print("restored", restore_from_drive(), "files from Drive")
os.makedirs(os.path.join(OUT, "amber"), exist_ok=True)
for z in glob.glob(f"{CACHE}/*.zip"):                       # AMBER zip saved by an earlier session
    if not os.path.exists(os.path.join(OUT, "amber", os.path.basename(z))):
        shutil.copy(z, os.path.join(OUT, "amber"))
!python scripts/phase5.py setup  --out "$OUT"
!python scripts/phase5.py images --out "$OUT"
for z in glob.glob(f"{OUT}/amber/*.zip"):                   # keep the AMBER zip for next time
    if not os.path.exists(os.path.join(CACHE, os.path.basename(z))):
        shutil.copy(z, CACHE)
p = json.load(open(os.path.join(OUT, "prep5.json")))
print("calibration", len(p["cal"]), "| test", len(p["test"]), "| AMBER", len(p["amber"]))
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout)

In [ ]:
# 5. Smoke test: 2 images per generator in a scratch folder (skipped once results exist)
SMOKE = os.path.join(WORK, "phase5_smoke")
if glob.glob(os.path.join(OUT, "captions5_*.s*.jsonl")):
    print("Skipping the smoke test: results already exist.")
else:
    os.makedirs(os.path.join(SMOKE, "amber"), exist_ok=True)
    for f in ["synonyms.txt", "prep5.json", "prep.json"]:
        shutil.copy(os.path.join(OUT, f), os.path.join(SMOKE, f))
    shutil.copy(os.path.join(OUT, "amber", "annotations.json"), os.path.join(SMOKE, "amber", "annotations.json"))
    for src, dst in [(os.path.join(OUT, "images"), os.path.join(SMOKE, "images")),
                     (os.path.join(OUT, "amber", "images"), os.path.join(SMOKE, "amber", "images"))]:
        if not os.path.exists(dst):
            os.symlink(src, dst)
    !python scripts/phase5.py captions --out "$SMOKE" --gens "$GENS" --limit 2
    !python scripts/phase5.py detect   --out "$SMOKE" --gens "$GENS"
    for f in sorted(glob.glob(os.path.join(SMOKE, "captions5_*_coco.s0.jsonl"))):
        r = json.loads(open(f).readline())
        print(os.path.basename(f), "->", r["caption"][:160])
    print("OWLv2 rows:", sum(1 for f in glob.glob(os.path.join(SMOKE, "owl5_*.jsonl")) for _ in open(f)))

In [ ]:
# 6. GPU work: each generator in turn (captions), then the detectors. Saved to Drive every 5 minutes.
LOG = os.path.join(OUT, "gpu5_0.log")
t0 = time.time()

def free_disk_gb():
    return shutil.disk_usage("/content").free / 1e9

def run(args, label):
    """Run one phase5.py stage in its own process; True if it finished, False if time ran out."""
    print(time.strftime("%H:%M"), "start", label, flush=True)
    with open(LOG, "a") as log:
        p = subprocess.Popen([sys.executable, "scripts/phase5.py", *args], stdout=log, stderr=subprocess.STDOUT)
        last = time.time()
        while p.poll() is None:
            time.sleep(30)
            if time.time() - last > 300:
                save_to_drive(); last = time.time()
                tail = open(LOG).readlines()[-1:]                    # show it is still working
                print(time.strftime("%H:%M"), tail[0].strip() if tail else "starting", flush=True)
            if time.time() - t0 > SESSION_HOURS * 3600:
                p.terminate(); p.wait()
                print("Session time limit reached: stopping (finished items are saved).")
                return False
    save_to_drive()
    print(time.strftime("%H:%M"), "done ", label, "| exit", p.returncode, "| free disk", round(free_disk_gb()), "GB")
    if p.returncode:
        print("".join(open(LOG).readlines()[-8:]))
    return True

ok = True
for g in GENS.split(","):
    if not ok:
        break
    ok = run(["captions", "--out", OUT, "--gens", g, *LIMIT], f"captions {g}")
    if free_disk_gb() < 25:                       # model weights fill the disk: drop the ones already used
        shutil.rmtree(os.path.join(os.environ["HF_HOME"], "hub"), ignore_errors=True)
        print("cleared model cache, free disk now", round(free_disk_gb()), "GB")
if ok:
    run(["detect", "--out", OUT, "--gens", GENS], "detectors (OWLv2, OWL-ViT, Grounding DINO)")
print("GPU work stopped after", round((time.time() - t0) / 3600, 2), "h")
print("--- last log lines\n" + "".join(open(LOG).readlines()[-6:]))

In [ ]:
# 7. Progress check
sys.path.insert(0, "scripts")
import importlib, phase5 as P5, phase4 as P4
from pathlib import Path
importlib.reload(P5)
OUTP = Path(OUT)
p = P5._prep(OUTP)
n_coco = len(p["cal"]) + len(p["test"]) if MODE == "full" else 100
n_amber = len(p["amber"]) if MODE == "full" else 100
COMPLETE = True
for g in GENS.split(","):
    for ds, want in (("coco", n_coco), ("amber", n_amber)):
        caps = len(P4.read_all(OUTP, P5.cap_file(g, ds)))
        owl = len(P4.read_all(OUTP, f"owl5_{g}_{ds}"))
        swap = len(P4.read_all(OUTP, f"swap5_{g}_{ds}"))
        ok = caps >= want and owl >= 0.97 * caps and swap >= 0.97 * caps
        COMPLETE &= ok
        print(f"{g:13s} {ds:6s} captions {caps:5d}/{want:<5d} owl {owl:5d} swap {swap:5d} {'done' if ok else ''}")
print("COMPLETE" if COMPLETE else "Not complete: run the notebook again in a new session.")

In [ ]:
# 8. Pilot check: CHAIR and truncation per generator. Send this output for review before Wave 1.
!python scripts/phase5.py pilot --out "$OUT" --gens $GENS

In [ ]:
# 9. Analysis and audit sample (CPU). Runs only when the wave is complete.
#    On full data this must run from the frozen pre-registration (REF = "prereg-v3").
ALL = ",".join(g for g in WAVES[1].split(",") + WAVES[2].split(",")
               if P4.read_all(OUTP, P5.cap_file(g, "coco")))
if MODE == "pilot":
    print("Pilot mode: no analysis (the pilot images are calibration images). Check cell 8 instead.")
elif not COMPLETE:
    print("Not complete yet: skipping the analysis.")
elif REF != "prereg-v3":
    print("Not running the confirmatory analysis: freeze the pre-registration and set REF = 'prereg-v3' first.")
else:
    !python scripts/phase5.py analyze --out "$OUT" --gens "$ALL"
    !python scripts/phase5.py audit --out "$OUT" --gens "llava,qwen25,internvl3"
    r = json.load(open(os.path.join(OUT, "phase5_results.json")))
    for s, row in r["matrices"]["owlv2"].items():
        print(s, {t: (round(c["claim"]["point"], 3), c["claim"]["decision"][:4]) for t, c in row.items()})

In [ ]:
# 10. Package results (no images) and save everything to Drive
import zipfile
zpath = os.path.join(OUT, "phase5_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in (glob.glob(f"{OUT}/*.jsonl") + glob.glob(f"{OUT}/*.log") + glob.glob(f"{OUT}/audit/*.csv")
              + [f"{OUT}/prep5.json", f"{OUT}/phase5_results.json"]):
        if os.path.exists(f):
            z.write(f, os.path.relpath(f, OUT))
    z.writestr("environment.txt", f"platform colab | gpu {torch.cuda.get_device_name(0)}\n"
               f"torch {torch.__version__}\ntransformers {transformers.__version__}\n"
               + subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("copied", save_to_drive(), "files to Drive")
print("Saved:", os.path.join(SAVE, "phase5_results.zip"), round(os.path.getsize(zpath) / 1e6, 1), "MB |",
      "COMPLETE" if COMPLETE else "partial")